In [1]:
import pandas as pd
import os
import numpy as np
import re

/Users/tracyliu/opt/anaconda3/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


### A. DICOM

In [2]:
file_path = "../Data/dicom.xlsx"
dicom = pd.read_excel(file_path)

In [3]:
dicom[dicom["PatientID"]==4330018595]

,PatientID,AccessionNumber,PatientBirthDate,PatientAge,PatientSex,StudyDate,StudyTime,AcquisitionDate,Modality,Manufacturer,ManufacturerModelName,StudyDescription,SeriesNumber,SeriesDescription,Exposure,Rows,Columns,PixelSpacing
0,4330018595,60103700,1965-07-01,054Y,F,2020-06-02,09:10:52,NaN,MG,"R2 Technology, Inc.",Cenova,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,1.0,Hologic R2 ImageChecker CAD SC,NaN,1500.0,1250.0,NaN
1,4330018595,60103700,1965-07-01,NaN,F,2020-06-02,09:10:51,NaN,MG,"R2 Technology, Inc.",Cenova,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,1.0,Hologic R2 ImageChecker CAD SC,NaN,1500.0,1250.0,NaN
2,4330018595,60103700,1965-07-01,NaN,F,2020-06-02,09:10:51,2020-06-02,MG,"HOLOGIC, Inc.",Selenia Dimensions,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,71100000.0,R ML,126.0,3328.0,2560.0,0.038889\0.038889
3,4330018595,60103700,1965-07-01,NaN,F,2020-06-02,09:10:51,NaN,MG,"HOLOGIC, Inc.",Selenia Dimensions,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,71300000.0,R XCCL Intelligent 2D,63.0,3328.0,2560.0,0.064318\0.064318
4,4330018595,60103700,1965-07-01,NaN,F,2020-06-02,09:10:51,NaN,MG,"HOLOGIC, Inc.",Selenia Dimensions,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,71300000.0,R ML Intelligent 2D,57.0,3328.0,2560.0,0.064619\0.064619
5,4330018595,60103700,1965-07-01,NaN,F,2020-06-02,09:10:52,2020-06-02,MG,"HOLOGIC, Inc.",Selenia Dimensions,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,71100000.0,R XCCL,78.0,3328.0,2560.0,0.038889\0.038889
6,4330018595,60103700,1965-07-01,NaN,F,2020-06-02,09:10:52,NaN,MG,"HOLOGIC, Inc.",Selenia Dimensions,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,71300000.0,R MLO Intelligent 2D,63.0,3328.0,2560.0,0.064217\0.064217
7,4330018595,60103700,1965-07-01,NaN,F,2020-06-02,09:10:52,NaN,MG,"HOLOGIC, Inc.",Selenia Dimensions,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,71300000.0,R CC Intelligent 2D,61.0,3328.0,2560.0,0.064519\0.064519
8,4330018595,60103700,1965-07-01,NaN,F,2020-06-02,09:10:51,2020-06-02,MG,"HOLOGIC, Inc.",SecurView,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,74100000.0,SecurView Secondary Capture,NaN,2456.0,1889.0,0.087154\0.087154
9,4330018595,60103700,1965-07-01,NaN,F,2020-06-02,09:10:51,2020-06-02,MG,"HOLOGIC, Inc.",SecurView,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,74100000.0,SecurView Secondary Capture,NaN,2456.0,1889.0,0.0874264\0.0874264


In [4]:
dicom["PatientID"].unique().size

5515

In [5]:
dicom["Modality"].unique()

array(['MG', 'PR', 'SR', 'US', 'MR', 'OT', 'KO', 'SC'], dtype=object)

In [6]:
laterality_bi = ["BILATERAL", "BILAT", "BIL", "BI"]
laterality_uni = ["UNILATERAL", "UNILAT", "UNI", "Unilateral"]
other = ["IMPLANT", "CONTRAST ENHANCED", "Implant"]
modality = ["TOMOSYNTHESIS", "TOMOSYN", "TOMO", "TOMOHD", "TomoHD"]

In [7]:
PIDs = dicom["PatientID"].unique()

In [8]:
dicom_copy = dicom.copy()

#### Study

In [9]:
study_screen = ["SCREENING", "SCREEN", "Screening"]
study_diag = ["DIAG", "DIAGNOSTIC", "DX", "DIG", "Diagnostic"]

In [10]:
# Assuming your DataFrame is named 'df' and the column where you found the words is 'Notes'
column_to_check = 'StudyDescription' 
type_column = 'Study'

# Ensure the column is treated as strings and handle missing values
dicom_copy[column_to_check] = dicom_copy[column_to_check].astype(str)

# ---Create the OR-Condition Mask ---
# The pattern uses the '|' (OR) symbol.
# We use .str.contains() with case=False to handle both "DIAG" and "Diagnostic".
diag_terms = [term for term in study_diag]
diag_pattern = '|'.join([re.escape(term) for term in diag_terms])

screen_terms = [term for term in study_screen]
screen_pattern = '|'.join([re.escape(term) for term in screen_terms])

# Create a boolean mask: True if the string contains either word (case-insensitive)
diag_mask = dicom_copy[column_to_check].str.contains(diag_pattern, case=False, na=False, regex=True)
screen_mask = dicom_copy[column_to_check].str.contains(screen_pattern, case=False, na=False, regex=True)


# --- Assign the value "DIAG" to the 'type' column ---
# Use .loc to assign the value only where the mask is True
dicom_copy.loc[diag_mask, type_column] = "DIAG"
dicom_copy.loc[screen_mask, type_column] = "SCREEN"

#### Side

In [11]:
side_right = ["RIGHT", "RT", "Right", "R XCCL", "R MLO", "R CC", "R ML", "R SIO"]
side_left = ["LEFT", "LT", "Left", "L XCCL", "L MLO", "L CC", "L ML", "L SIO"]

In [12]:
# Assuming your DataFrame is named 'df' and the column where you found the words is 'Notes'
column_to_check = 'SeriesDescription' 
type_column = 'Side'

# Ensure the column is treated as strings and handle missing values
dicom_copy[column_to_check] = dicom_copy[column_to_check].astype(str)

# ---Create the OR-Condition Mask ---
# The pattern uses the '|' (OR) symbol.
# We use .str.contains() with case=False to handle both "DIAG" and "Diagnostic".
side_right_terms = [term for term in side_right]
side_right_pattern = '|'.join([re.escape(term) for term in side_right_terms])

side_left_terms = [term for term in side_left]
side_left_pattern = '|'.join([re.escape(term) for term in side_left_terms])

# Create a boolean mask: True if the string contains either word (case-insensitive)
side_right_mask = dicom_copy[column_to_check].str.contains(side_right_pattern, case=False, na=False, regex=True)
side_left_mask = dicom_copy[column_to_check].str.contains(side_left_pattern, case=False, na=False, regex=True)


# --- Assign the value "DIAG" to the 'type' column ---
# Use .loc to assign the value only where the mask is True
dicom_copy.loc[side_right_mask, type_column] = "R"
dicom_copy.loc[side_left_mask, type_column] = "L"

In [13]:
dicom_copy

,PatientID,AccessionNumber,PatientBirthDate,PatientAge,PatientSex,StudyDate,StudyTime,AcquisitionDate,Modality,Manufacturer,ManufacturerModelName,StudyDescription,SeriesNumber,SeriesDescription,Exposure,Rows,Columns,PixelSpacing,Study,Side
0,4330018595,60103700,1965-07-01,054Y,F,2020-06-02,09:10:52,NaN,MG,"R2 Technology, Inc.",Cenova,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,1.0,Hologic R2 ImageChecker CAD SC,NaN,1500.0,1250.0,NaN,DIAG,NaN
1,4330018595,60103700,1965-07-01,NaN,F,2020-06-02,09:10:51,NaN,MG,"R2 Technology, Inc.",Cenova,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,1.0,Hologic R2 ImageChecker CAD SC,NaN,1500.0,1250.0,NaN,DIAG,NaN
2,4330018595,60103700,1965-07-01,NaN,F,2020-06-02,09:10:51,2020-06-02,MG,"HOLOGIC, Inc.",Selenia Dimensions,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,71100000.0,R ML,126.0,3328.0,2560.0,0.038889\0.038889,DIAG,R
3,4330018595,60103700,1965-07-01,NaN,F,2020-06-02,09:10:51,NaN,MG,"HOLOGIC, Inc.",Selenia Dimensions,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,71300000.0,R XCCL Intelligent 2D,63.0,3328.0,2560.0,0.064318\0.064318,DIAG,R
4,4330018595,60103700,1965-07-01,NaN,F,2020-06-02,09:10:51,NaN,MG,"HOLOGIC, Inc.",Selenia Dimensions,DIAG DIG MAMMO RIGHT ALL VIEWS WITH TOMOSYNTHESIS,71300000.0,R ML Intelligent 2D,57.0,3328.0,2560.0,0.064619\0.064619,DIAG,R
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
136228,4339959661,62216536,1971-07-01,NaN,F,2019-10-28,02:45:37,2019-10-28,MG,"HOLOGIC, Inc.",Selenia Dimensions,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,72100000.0,L CC Tomosynthesis Projection,72.0,425.0,266.0,NaN,SCREEN,L
136229,4339959661,62216536,1971-07-01,NaN,F,2019-10-28,02:45:37,2019-10-28,MG,"HOLOGIC, Inc.",Selenia Dimensions,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,72100000.0,R CC Tomosynthesis Projection,74.0,425.0,266.0,NaN,SCREEN,R
136230,4339959661,62216536,1971-07-01,NaN,F,2019-10-28,02:45:37,2019-10-28,MG,"HOLOGIC, Inc.",Selenia Dimensions,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,72100000.0,L MLO Tomosynthesis Projection,78.0,425.0,266.0,NaN,SCREEN,L
136231,4339959661,62216536,1971-07-01,NaN,F,2019-10-28,18:20:08,NaN,PR,Philips Medical Systems,iSite Enterprise,SCREENING MAMMOGRAPHY DIGITAL BILATERAL W TOMOSYN,1.0,5DBAD14E0,NaN,NaN,NaN,NaN,SCREEN,NaN


#### SAVE

In [14]:
output_file = os.path.join("../Data/",'dicom_tag' + ".xlsx")
dicom_copy.to_excel(output_file, index=False)

<ipython-input-14-d6f02184cf18>:2: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.3.7' currently installed).
  dicom_copy.to_excel(output_file, index=False)


#### TEST

In [2]:
shared_path = "../Data/Lee, Ju Hun's files - R3Data"

In [4]:
# study = "Control"
# folder = ["R3_3787_Lee_Control_Extract_Files", "R3_3787_Lee_Data_Controls_20240508"]
# study_ext = "_controls"

study = "Cancer"
folder = ["R3_3787_Lee_Cancer_Extract_Files", "R3_3787_Lee_Data_Cancer_20240509", "R3_3787_Lee_Data_Cancer_20250912"]
study_ext = ""

In [14]:
file_name = "procedure_notes"
file_path = os.path.join(shared_path, study, folder[1], file_name + ".csv")
df = pd.read_csv(file_path, encoding='latin-1')

<ipython-input-14-a2e0786f4363>:3: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, encoding='latin-1')


In [15]:
df

,PATIENT_STUDY_ID,ORDER_DATE,RESULT_TIME,PROCEDURE_CODE,PROCEDURE_NAME,NOTE_CSN_ID,LINE_NUM,NOTE_TEXT
0,4330114580,07/11/2017,07/17/2017 09:15:00,76140,Consultation on X-ray examination made elsewhe...,990311450711,1,CLINICAL HISTORY
1,4330114580,07/11/2017,07/17/2017 09:15:00,76140,Consultation on X-ray examination made elsewhe...,990311450711,2,29 year old female recently diagnosed (06/2017...
2,4330114580,07/11/2017,07/17/2017 09:15:00,76140,Consultation on X-ray examination made elsewhe...,990311450711,3,"IA (T1, N0, M0) invasive ductal carcinoma, Nuc..."
3,4330114580,07/11/2017,07/17/2017 09:15:00,76140,Consultation on X-ray examination made elsewhe...,990311450711,4,"receptor negative, progesterone receptor negat..."
4,4330114580,07/11/2017,07/17/2017 09:15:00,76140,Consultation on X-ray examination made elsewhe...,990311450711,5,equivocal FISH pending and Ki-67 approximately...
...,...,...,...,...,...,...,...,...
1519548,4339886616,08/03/2018,08/03/2018 17:22:00,76098,"Radiological examination, surgical specimen",990365639883,3,FINDINGS/IMPRESSION: After surgery on the lef...
1519549,4339886616,08/03/2018,08/03/2018 17:22:00,76098,"Radiological examination, surgical specimen",990365639883,4,surgical specimen was brought to the Radiology...
1519550,4339886616,08/03/2018,08/03/2018 17:22:00,76098,"Radiological examination, surgical specimen",990365639883,5,demonstrates one radioactive seed and numerous...
1519551,4339886616,08/03/2018,08/03/2018 17:22:00,76098,"Radiological examination, surgical specimen",990365639883,6,U-shaped clip is not in the specimen. The resu...


### Files

See **parameter explanation for detail information

In [6]:
file_path = "../Data/parameters of interest.xlsx"
files = pd.ExcelFile(file_path).sheet_names